# 05. Sensor Individual Analysis
## Structural Health Monitoring — Heritage Masonry Vaults

**Purpose:** Generate the full visual and statistical suite for each sensor individually.
This notebook is **parameterized** — changing `sensor` in Section 0 and re-running
produces the complete analysis for that sensor. All 12 sensors use identical logic.

**Input:** `../data/processed/clean_{sensor}.csv` — individual clean CSV from `02_data_cleaning.ipynb`

**Output per sensor (saved to `../outputs/figures/{sensor}/`):**
- `{sensor}_01_displacement_distribution.png` — relative displacement histogram with percentile bands
- `{sensor}_02_displacement_ranges_barchart.png` — bar chart by percentile range
- `{sensor}_03_absolute_values_histogram.png` — histogram of raw absolute values
- `{sensor}_04_temporal_evolution.png` — time series with IQR reference lines
- `{sensor}_05_temporal_vs_environment.png` — normalized overlay vs environmental variables
- `{sensor}_06_correlation_matrix.png` — Pearson heatmap vs environment
- `{sensor}_07_analysis_table.xlsx` — percentile range table (two sheets: absolute + relative)

**To run for all sensors at once:** use Section 6 (full batch loop).

**Next:** `INFRASTRUCTURE_ANALYSIS_REPORT.md` — anonymized technical report with all visualizations.

---
## 0. Configuration

In [19]:
# ============================================================
# IMPORTS
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

from sklearn.preprocessing import MinMaxScaler
from datetime import datetime

# ============================================================
# SENSOR CONFIGURATION
# ============================================================
SENSOR_CONFIG = {
    'S01': {'zone': 'Zone A', 'axis': '(Diag.)', 'color': '#0563C1'},
    'S02': {'zone': 'Zone A', 'axis': '(Hor.)',  'color': '#ADD8E6'},
    'S03': {'zone': 'Zone A', 'axis': '(Diag.)', 'color': '#0563C1'},
    'S04': {'zone': 'Zone A', 'axis': '(Hor.)',  'color': '#ADD8E6'},
    'S05': {'zone': 'Zone B', 'axis': '(Diag.)', 'color': '#FFA500'},
    'S06': {'zone': 'Zone B', 'axis': '(Hor.)',  'color': '#FFD580'},
    'S07': {'zone': 'Zone B', 'axis': '(Diag.)', 'color': '#FFA500'},
    'S08': {'zone': 'Zone B', 'axis': '(Hor.)',  'color': '#FFD580'},
    'S09': {'zone': 'Zone C', 'axis': '(Diag.)', 'color': '#6EAB54'},
    'S10': {'zone': 'Zone C', 'axis': '(Hor.)',  'color': '#B6D5A9'},
    'S11': {'zone': 'Zone C', 'axis': '(Diag.)', 'color': '#6EAB54'},
    'S12': {'zone': 'Zone C', 'axis': '(Hor.)',  'color': '#B6D5A9'},
}

SENSORS = list(SENSOR_CONFIG.keys())

# ============================================================
# PATHS
# ============================================================
PROCESSED_DATA_PATH = '../data/processed/'
FIGURES_PATH        = '../outputs/figures/'
TABLES_PATH         = '../outputs/tables/'

os.makedirs(TABLES_PATH, exist_ok=True)

# ============================================================
# ACTIVE SENSOR
# Change this to run the full analysis for any single sensor.
# To process all 12 sensors at once: go to Section 6.
# ============================================================
sensor = 'S01'   # <-- CHANGE THIS (S01 through S12)

config     = SENSOR_CONFIG[sensor]
sensor_dir = os.path.join(FIGURES_PATH, sensor)
os.makedirs(sensor_dir, exist_ok=True)

print(f'Active sensor: {sensor} | Zone: {config["zone"]} | Axis: {config["axis"]}')
print(f'Output directory: {sensor_dir}')

Active sensor: S01 | Zone: Zone A | Axis: (Diag.)
Output directory: ../outputs/figures/S01


---
## 1. BATCH MODE - ALL 12 SENSORS

In [20]:
# ============================================================
# BATCH MODE — ALL 12 SENSORS
# Generates the complete visual suite for all 12 sensors.
# Outputs saved to ../outputs/figures/{sensor}/
# 6 files per sensor × 12 sensors = 72 files total
#
# Output files per sensor:
#   {sensor}_02_displacement_distribution.png  — absolute histogram, relative x-axis labels
#   {sensor}_01_displacement_ranges_barchart.png — bar chart by percentile range
#   {sensor}_03_temporal_evolution.png          — time series with IQR reference lines
#   {sensor}_04_temporal_vs_environment.png     — normalized overlay vs env variables
#   {sensor}_05_correlation_matrix.png          — Pearson heatmap vs environment
#   {sensor}_06_analysis_table.xlsx             — percentile range table (2 sheets)
# ============================================================
import matplotlib
matplotlib.use('Agg')   # non-interactive backend — no popup windows

batch_summary = []

for sensor in SENSORS:
    cfg   = SENSOR_CONFIG[sensor]
    s_dir = os.path.join(FIGURES_PATH, sensor)
    os.makedirs(s_dir, exist_ok=True)

    print(f"\n{'='*60}")
    print(f"Processing: {sensor} {cfg['axis']} | {cfg['zone']}")
    print(f"{'='*60}")

    try:
        df = pd.read_csv(os.path.join(PROCESSED_DATA_PATH, f'clean_{sensor}.csv'))
        df['Timestamp'] = pd.to_datetime(df['Timestamp'])
    except FileNotFoundError:
        print(f'  clean_{sensor}.csv not found — skipping')
        continue

    d         = df[sensor].dropna().copy()
    d_min     = d.min()
    d_max     = d.max()
    d_mov     = d - d_min
    mov_total = d_max - d_min

    p1_b    = float(d_mov.quantile(0.01))
    p2_5_b  = float(d_mov.quantile(0.025))
    p5_b    = float(d_mov.quantile(0.05))
    p10_b   = float(d_mov.quantile(0.10))
    Q1_b    = float(d_mov.quantile(0.25))
    p50_b   = float(d_mov.quantile(0.50))
    Q3_b    = float(d_mov.quantile(0.75))
    p90_b   = float(d_mov.quantile(0.90))
    p95_b   = float(d_mov.quantile(0.95))
    p97_5_b = float(d_mov.quantile(0.975))
    p99_b   = float(d_mov.quantile(0.99))

    # ── Plot 02: Displacement Distribution (absolute histogram, relative x-axis) ──
    # Uses absolute sensor values (d) for the histogram bars,
    # but x-axis ticks are labeled as relative displacement (mm from structural minimum).
    # Percentile lines use absolute positions (d_min + relative offset).
    fig, ax = plt.subplots(figsize=(14, 7))
    sns.histplot(d, bins=50, edgecolor='black', kde=True,
                 alpha=0.6, color=cfg['color'], label='Values', ax=ax)
    for v, lbl, col, sty in [
        (d_min+p1_b,  'P1',    'red',    ':'),
        (d_min+p5_b,  'P5',    'orange', '--'),
        (d_min+Q1_b,  'Q1',    'green',  '-'),
        (d_min+p50_b, 'Median','purple', '-'),
        (d_min+Q3_b,  'Q3',    'green',  '-'),
        (d_min+p95_b, 'P95',   'orange', '--'),
        (d_min+p99_b, 'P99',   'red',    ':')
    ]:
        ax.axvline(v, color=col, linestyle=sty, linewidth=2.5, alpha=0.8, label=lbl)
    margen = (d.max() - d.min()) * 0.05
    ax.set_xlim(d.min() - margen, d.max() + margen)
    ax.set_xticks([d_min+p1_b, d_min+p5_b, d_min+Q1_b, d_min+p50_b,
                   d_min+Q3_b, d_min+p95_b, d_min+p99_b])
    ax.set_xticklabels([f'P1\n{p1_b:.1f}mm', f'P5\n{p5_b:.1f}mm',
                        f'Q1\n{Q1_b:.1f}mm',  f'Median\n{p50_b:.1f}mm',
                        f'Q3\n{Q3_b:.1f}mm',  f'P95\n{p95_b:.1f}mm',
                        f'P99\n{p99_b:.1f}mm'],
                       fontsize=10, fontweight='bold')
    ax.set_xlabel('Relative Displacement (mm)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Frequency', fontsize=12, fontweight='bold')
    ax.set_title(f'Displacement Distribution — {sensor} {cfg["axis"]} | {cfg["zone"]}\n'
                 f'Monitoring data — April 2025 to February 2026',
                 fontsize=13, fontweight='bold')
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig(os.path.join(s_dir, f'{sensor}_02_displacement_distribution.png'),
                dpi=300, bbox_inches='tight')
    plt.close()
    print(f'  01 done')

    # ── Plot 01: Displacement ranges bar chart ────────────────
    rangos_data = []
    rango_1 = len(d_mov[d_mov <= p1_b])
    rangos_data.append({'Range': 'Min → P1', 'Lower': 0.0, 'Upper': p1_b,
                        'Count': rango_1, 'Pct': rango_1/len(d_mov)*100,
                        'Cum': rango_1/len(d_mov)*100})
    intermedios = [
        ('P1',p1_b,'P2.5',p2_5_b),('P2.5',p2_5_b,'P5',p5_b),('P5',p5_b,'P10',p10_b),
        ('P10',p10_b,'Q1',Q1_b),('Q1',Q1_b,'P50',p50_b),('P50',p50_b,'Q3',Q3_b),
        ('Q3',Q3_b,'P90',p90_b),('P90',p90_b,'P95',p95_b),
        ('P95',p95_b,'P97.5',p97_5_b),('P97.5',p97_5_b,'P99',p99_b)
    ]
    cum = rango_1
    for n_inf, v_inf, n_sup, v_sup in intermedios:
        n = len(d_mov[(d_mov > v_inf) & (d_mov <= v_sup)])
        cum += n
        rangos_data.append({'Range': f'{n_inf} → {n_sup}', 'Lower': v_inf, 'Upper': v_sup,
                            'Count': n, 'Pct': n/len(d_mov)*100, 'Cum': cum/len(d_mov)*100})
    n_final = len(d_mov[d_mov > p99_b])
    rangos_data.append({'Range': 'P99 → Max', 'Lower': p99_b, 'Upper': mov_total,
                        'Count': n_final, 'Pct': n_final/len(d_mov)*100, 'Cum': 100.0})
    df_rangos_dif = pd.DataFrame(rangos_data)

    colores = []
    for i, r in enumerate(df_rangos_dif['Range']):
        if i == 0 or i == len(df_rangos_dif) - 1:
            colores.append('#ff6b6b')
        elif 'P1' in r or 'P99' in r or 'P97.5' in r or 'P2.5' in r:
            colores.append('#ffa500')
        elif 'Q1' in r or 'P50' in r or 'Q3' in r:
            colores.append('#2ecc71')
        else:
            colores.append(cfg['color'])

    fig, ax = plt.subplots(figsize=(20, 10))
    barras = ax.bar(range(len(df_rangos_dif)), df_rangos_dif['Count'],
                    color=colores, edgecolor='black', linewidth=1.5, alpha=0.85)
    for barra, row in zip(barras, df_rangos_dif.itertuples()):
        ax.text(barra.get_x() + barra.get_width()/2.,
                barra.get_height() + df_rangos_dif['Count'].max()*0.01,
                f'{row.Pct:.1f}%\n({int(row.Count)})',
                ha='center', va='bottom', fontsize=11, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.8, edgecolor='black'))
    etiquetas = [f"{r['Range']}\n({r['Lower']:.1f}→{r['Upper']:.1f}mm)"
                 for _, r in df_rangos_dif.iterrows()]
    ax.set_xticks(range(len(etiquetas)))
    ax.set_xticklabels(etiquetas, rotation=45, ha='right', fontsize=10, fontweight='bold')
    ax.set_xlabel('Displacement Ranges (mm)', fontsize=13, fontweight='bold')
    ax.set_ylabel('Record Count', fontsize=13, fontweight='bold')
    ax.set_title(f'Displacement Ranges — {sensor} {cfg["axis"]} | {cfg["zone"]}\n'
                 f'Monitoring data — April 2025 to February 2026',
                 fontsize=14, fontweight='bold', pad=20)
    ax.grid(True, alpha=0.3, axis='y')
    plt.subplots_adjust(top=0.95, bottom=0.15, left=0.08, right=0.98)
    plt.tight_layout()
    plt.savefig(os.path.join(s_dir, f'{sensor}_01_displacement_ranges_barchart.png'),
                dpi=300, bbox_inches='tight')
    plt.close()
    print(f'  01 done')

    # ── Plot 03: Temporal evolution ───────────────────────────
    colchon = mov_total * 0.1
    fig, ax = plt.subplots(figsize=(16, 7))
    ax.plot(df['Timestamp'], df[sensor], linewidth=1.5, alpha=0.8,
            color=cfg['color'], label='Temporal evolution')
    ax.set_ylim(d_min - colchon, d_max + colchon)
    for v, lbl, col in [
        (d_min+p5_b,  f'P5:{d_min+p5_b:.1f}',    'yellow'),
        (d_min+Q1_b,  f'Q1:{d_min+Q1_b:.1f}',    'green'),
        (d_min+p50_b, f'Median:{d_min+p50_b:.1f}','purple'),
        (d_min+Q3_b,  f'Q3:{d_min+Q3_b:.1f}',    'green'),
        (d_min+p95_b, f'P95:{d_min+p95_b:.1f}',  'yellow')
    ]:
        ax.axhline(v, color=col, linestyle='--', linewidth=2.5, alpha=0.7, label=lbl)
    info = (f"INFO {sensor} | Period: {df['Timestamp'].min()} to {df['Timestamp'].max()}\n"
            f"Total records: {len(df)} | Min: {d_min:.2f} | Max: {d_max:.2f}\n"
            f"Mean: {d.mean():.2f} | Std.Dev: {d.std():.2f} | Range: {mov_total:.2f}mm")
    ax.text(0.05, -0.22, info, transform=ax.transAxes, fontsize=9,
            va='top', ha='left', family='monospace', fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='#E8E8E8', alpha=0.9))
    ax.set_title(f'Temporal Evolution — {sensor} {cfg["axis"]} | {cfg["zone"]}\n'
                 f'Monitoring data — April 2025 to February 2026',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('Timestamp', fontsize=12)
    ax.set_ylabel('Values (mm)', fontsize=12)
    ax.legend(fontsize=10, loc='upper left')
    ax.grid(True, alpha=0.3)
    plt.subplots_adjust(bottom=0.18)
    plt.tight_layout()
    plt.savefig(os.path.join(s_dir, f'{sensor}_03_temporal_evolution.png'),
                dpi=300, bbox_inches='tight')
    plt.close()
    print(f'  03 done')

    # ── Plot 04: Temporal vs environmental overlay ────────────
    df_norm = df.copy()
    scaler_s = MinMaxScaler(feature_range=(0, 10))
    df_norm[sensor] = scaler_s.fit_transform(df[[sensor]])
    for var in ['Temp.', 'Hum.', 'Pres.']:
        scaler_v = MinMaxScaler(feature_range=(0, 10))
        df_norm[var] = scaler_v.fit_transform(df[[var]].fillna(df[var].mean()))
    fig, ax = plt.subplots(figsize=(40, 6))
    ax.plot(df_norm['Timestamp'], df_norm[sensor],
            label=sensor, linewidth=2.5, color=cfg['color'], alpha=0.9)
    ax.plot(df_norm['Timestamp'], df_norm['Temp.'],
            label='Temp. (normalized)', linewidth=1.5, alpha=0.4, color='red')
    ax.plot(df_norm['Timestamp'], df_norm['Hum.'],
            label='Hum. (normalized)', linewidth=1.5, alpha=0.6, color='lightblue')
    ax.plot(df_norm['Timestamp'], df_norm['Pres.'],
            label='Pres. (normalized)', linewidth=1.5, alpha=0.6, color='lightgreen')
    ax.set_title(f'{sensor} {cfg["axis"]} | {cfg["zone"]} vs Environmental Variables (Normalized 0–10)\n'
                 f'Monitoring data — April 2025 to February 2026 — Clean data',
                 fontsize=14, fontweight='bold')
    ax.set_xlabel('Time')
    ax.set_ylabel('Normalized Value (0–10)')
    ax.legend(loc='upper right', fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(s_dir, f'{sensor}_04_temporal_vs_environment.png'),
                dpi=300, bbox_inches='tight')
    plt.close()
    print(f'  04 done')

    # ── Plot 05: Correlation matrix ───────────────────────────
    corr_data = df[[sensor, 'Temp.', 'Hum.', 'Pres.']].corr()
    plt.figure(figsize=(8, 6))
    sns.heatmap(corr_data, annot=True, fmt='.2f',
                cmap=['#001F3F', '#FFFFFF', '#C41E3A'],
                center=0, square=True, linewidths=2,
                cbar_kws={'label': 'Correlation'})
    plt.title(f'Correlation Matrix — {sensor} {cfg["axis"]} | {cfg["zone"]}\n'
              f'vs Environmental Variables — Clean data\n'
              f'Monitoring data — April 2025 to February 2026',
              fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(s_dir, f'{sensor}_05_correlation_matrix.png'),
                dpi=300, bbox_inches='tight')
    plt.close()
    print(f'  05 done')

    # ── Export 06: Percentile range table xlsx ────────────────
    df_rangos_rel = df_rangos_dif[['Range','Lower','Upper','Count','Pct','Cum']].copy()
    df_rangos_rel.columns = ['Range','Lower (mm)','Upper (mm)','Count','% of total','Cumulative %']
    rangos_abs = []
    for _, row in df_rangos_dif.iterrows():
        rangos_abs.append({
            'Range'       : row['Range'],
            'Lower (mm)'  : round(d_min + row['Lower'], 2),
            'Upper (mm)'  : round(d_min + row['Upper'], 2),
            'Count'       : row['Count'],
            '% of total'  : round(row['Pct'], 2),
            'Cumulative %': round(row['Cum'], 2)
        })
    df_rangos_abs = pd.DataFrame(rangos_abs)
    fname_xlsx = f'{sensor}_06_analysis_table.xlsx'
    with pd.ExcelWriter(os.path.join(s_dir, fname_xlsx), engine='openpyxl') as writer:
        df_rangos_rel.to_excel(writer, sheet_name='Relative Displacement', index=False)
        df_rangos_abs.to_excel(writer, sheet_name='Absolute Values',       index=False)
    print(f'  06 done')

    batch_summary.append({
        'Sensor'              : sensor,
        'Zone'                : cfg['zone'],
        'Axis'                : cfg['axis'],
        'Records'             : len(d),
        'Max displacement (mm)': round(mov_total, 2),
        'Mean (mm)'           : round(d_mov.mean(), 2),
        'Std Dev (mm)'        : round(d_mov.std(), 2),
        'CV (%)'              : round(d_mov.std()/d_mov.mean()*100, 2),
        'IQR range (mm)'      : f'{Q1_b:.2f}–{Q3_b:.2f}',
        'Output folder'       : s_dir
    })
    print(f'  ✓ {sensor} complete — 6 files saved')

# Restore interactive backend
matplotlib.use('TkAgg')

df_batch = pd.DataFrame(batch_summary)
print(f"\n{'='*100}")
print(f"BATCH COMPLETE — {len(batch_summary)} sensors processed")
print(f"{'='*100}\n")
print(df_batch.to_string(index=False))

df_batch.to_excel(os.path.join(TABLES_PATH, '05_batch_summary.xlsx'), index=False)
print(f'\nExported: 05_batch_summary.xlsx')



Processing: S01 (Diag.) | Zone A
  01 done
  01 done
  03 done
  04 done
  05 done
  06 done
  ✓ S01 complete — 6 files saved

Processing: S02 (Hor.) | Zone A
  01 done
  01 done
  03 done
  04 done
  05 done
  06 done
  ✓ S02 complete — 6 files saved

Processing: S03 (Diag.) | Zone A
  01 done
  01 done
  03 done
  04 done
  05 done
  06 done
  ✓ S03 complete — 6 files saved

Processing: S04 (Hor.) | Zone A
  01 done
  01 done
  03 done
  04 done
  05 done
  06 done
  ✓ S04 complete — 6 files saved

Processing: S05 (Diag.) | Zone B
  01 done
  01 done
  03 done
  04 done
  05 done
  06 done
  ✓ S05 complete — 6 files saved

Processing: S06 (Hor.) | Zone B
  01 done
  01 done
  03 done
  04 done
  05 done
  06 done
  ✓ S06 complete — 6 files saved

Processing: S07 (Diag.) | Zone B
  01 done
  01 done
  03 done
  04 done
  05 done
  06 done
  ✓ S07 complete — 6 files saved

Processing: S08 (Hor.) | Zone B
  01 done
  01 done
  03 done
  04 done
  05 done
  06 done
  ✓ S08 complete — 6 

---
# Note: 
### "The cells below are for single-sensor exploratory analysis. To inspect a specific sensor, change **sensor = 'S01'** in Section 0 and run the cells individually. These sections produce no output files."

---
## 1. Load Clean Data

In [10]:
# ============================================================
# LOAD CLEAN SENSOR DATA
# ============================================================
df_limpio = pd.read_csv(os.path.join(PROCESSED_DATA_PATH, f'clean_{sensor}.csv'))
df_limpio['Timestamp'] = pd.to_datetime(df_limpio['Timestamp'])

# Extract clean sensor series
datos = df_limpio[sensor].dropna().copy()

# Relative displacement from structural minimum (most contracted state)
# Sensors measure absolute laser distance — structural variable of interest
# is movement relative to the minimum reference position
datos_min        = datos.min()
datos_max        = datos.max()
movimiento_total = datos_max - datos_min
datos_movimiento = datos - datos_min

# Compute full percentile set on relative displacement
p1    = float(datos_movimiento.quantile(0.01))
p2_5  = float(datos_movimiento.quantile(0.025))
p5    = float(datos_movimiento.quantile(0.05))
p10   = float(datos_movimiento.quantile(0.10))
Q1    = float(datos_movimiento.quantile(0.25))
p50   = float(datos_movimiento.quantile(0.50))
Q3    = float(datos_movimiento.quantile(0.75))
p90   = float(datos_movimiento.quantile(0.90))
p95   = float(datos_movimiento.quantile(0.95))
p97_5 = float(datos_movimiento.quantile(0.975))
p99   = float(datos_movimiento.quantile(0.99))

coef_var = datos_movimiento.std() / datos_movimiento.mean() * 100

print(f"""
=== STATISTICS — {sensor} {config['axis']} | {config['zone']} ===

Records: {len(datos)}
Period:  {df_limpio['Timestamp'].min()} to {df_limpio['Timestamp'].max()}

STRUCTURAL DISPLACEMENT:
  Maximum detected:      {movimiento_total:.2f} mm
  Average displacement:  {datos_movimiento.mean():.2f} mm
  Std. Dev.:             {datos_movimiento.std():.2f} mm
  Coeff. of Variation:   {coef_var:.2f}%

CHARACTERISTIC RANGES:
  50% central (IQR): {Q1:.2f} – {Q3:.2f} mm  (amplitude {Q3-Q1:.2f} mm)
  90% central:       {p5:.2f} – {p95:.2f} mm  (amplitude {p95-p5:.2f} mm)
  95% central:       {p2_5:.2f} – {p97_5:.2f} mm  (amplitude {p97_5-p2_5:.2f} mm)

ABSOLUTE VALUES:
  Min: {datos_min:.2f} mm   Max: {datos_max:.2f} mm   Mean: {datos.mean():.2f} mm
""")


=== STATISTICS — S12 (Diag.) | Zone A ===

Records: 3243
Period:  2025-04-01 13:00:00 to 2026-02-27 23:29:30

STRUCTURAL DISPLACEMENT:
  Maximum detected:      6.95 mm
  Average displacement:  2.09 mm
  Std. Dev.:             1.32 mm
  Coeff. of Variation:   63.23%

CHARACTERISTIC RANGES:
  50% central (IQR): 1.25 – 2.82 mm  (amplitude 1.57 mm)
  90% central:       0.39 – 4.75 mm  (amplitude 4.36 mm)
  95% central:       0.25 – 5.25 mm  (amplitude 5.00 mm)

ABSOLUTE VALUES:
  Min: 24643.75 mm   Max: 24650.70 mm   Mean: 24645.84 mm



---
## 2. Visualizations

In [11]:
# ============================================================
# PLOT 1: RELATIVE DISPLACEMENT HISTOGRAM
# Shows distribution of movement from structural minimum.
# Percentile lines mark the IQR, P95, and P99 bands.
# ============================================================
fig, ax = plt.subplots(figsize=(14, 7))

ax.hist(datos_movimiento, bins=50, edgecolor='black', alpha=0.6, color=config['color'])

for valor, label, color, style in [
    (p1,  'P1',    'red',    ':'),
    (p5,  'P5',    'orange', '--'),
    (Q1,  'Q1',    'green',  '-'),
    (p50, 'Median','purple', '-'),
    (Q3,  'Q3',    'green',  '-'),
    (p95, 'P95',   'orange', '--'),
    (p99, 'P99',   'red',    ':')
]:
    ax.axvline(valor, color=color, linestyle=style, linewidth=2.5, alpha=0.8, label=label)

margen = (datos_movimiento.max() - datos_movimiento.min()) * 0.05
ax.set_xlim(datos_movimiento.min() - margen, datos_movimiento.max() + margen)
ax.set_xticks([p1, p5, Q1, p50, Q3, p95, p99])
ax.set_xticklabels([f'P1\n{p1:.1f}mm', f'P5\n{p5:.1f}mm', f'Q1\n{Q1:.1f}mm',
                    f'Median\n{p50:.1f}mm', f'Q3\n{Q3:.1f}mm',
                    f'P95\n{p95:.1f}mm', f'P99\n{p99:.1f}mm'],
                   fontsize=10, fontweight='bold')
ax.set_xlabel('Relative Displacement (mm)', fontsize=12, fontweight='bold')
ax.set_ylabel('Frequency', fontsize=12, fontweight='bold')
ax.set_title(
    f'Displacement Distribution — {sensor} {config["axis"]} | {config["zone"]}\n'
    f'Monitoring data — April 2025 to February 2026\n'
    f'Structural Displacement Histogram',
    fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
ax.legend(loc='upper right', fontsize=10)
plt.tight_layout()
fname = f'{sensor}_01_displacement_distribution.png'
plt.savefig(os.path.join(sensor_dir, fname), dpi=300, bbox_inches='tight')
print(f'Exported: {fname}')
plt.show()

Exported: S12_01_displacement_distribution.png


In [12]:
# ============================================================
# PLOT 2: DISPLACEMENT RANGES BAR CHART
# Shows record count and percentage in each percentile band.
# Color coding: red = extremes, green = central IQR band,
# sensor color = intermediate bands.
# ============================================================
rangos_data = []
rango_1 = len(datos_movimiento[datos_movimiento <= p1])
rangos_data.append({'Range': 'Min → P1', 'Lower': 0.0, 'Upper': p1,
                    'Count': rango_1, 'Pct': rango_1/len(datos_movimiento)*100, 'Cum': rango_1/len(datos_movimiento)*100})

intermedios = [('P1',p1,'P2.5',p2_5),('P2.5',p2_5,'P5',p5),('P5',p5,'P10',p10),
               ('P10',p10,'Q1',Q1),('Q1',Q1,'P50',p50),('P50',p50,'Q3',Q3),
               ('Q3',Q3,'P90',p90),('P90',p90,'P95',p95),('P95',p95,'P97.5',p97_5),('P97.5',p97_5,'P99',p99)]

cum = rango_1
for n_inf, v_inf, n_sup, v_sup in intermedios:
    n = len(datos_movimiento[(datos_movimiento > v_inf) & (datos_movimiento <= v_sup)])
    cum += n
    rangos_data.append({'Range': f'{n_inf} → {n_sup}', 'Lower': v_inf, 'Upper': v_sup,
                        'Count': n, 'Pct': n/len(datos_movimiento)*100, 'Cum': cum/len(datos_movimiento)*100})

n_final = len(datos_movimiento[datos_movimiento > p99])
rangos_data.append({'Range': 'P99 → Max', 'Lower': p99, 'Upper': movimiento_total,
                    'Count': n_final, 'Pct': n_final/len(datos_movimiento)*100, 'Cum': 100.0})

df_rangos_dif = pd.DataFrame(rangos_data)

fig, ax = plt.subplots(figsize=(20, 10))

colores = []
for i, r in enumerate(df_rangos_dif['Range']):
    if i == 0 or i == len(df_rangos_dif) - 1:
        colores.append('#ff6b6b')
    elif 'P1' in r or 'P99' in r or 'P97.5' in r or 'P2.5' in r:
        colores.append('#ffa500')
    elif 'Q1' in r or 'P50' in r or 'Q3' in r:
        colores.append('#2ecc71')
    else:
        colores.append(config['color'])

barras = ax.bar(range(len(df_rangos_dif)), df_rangos_dif['Count'],
                color=colores, edgecolor='black', linewidth=1.5, alpha=0.85)

for i, (barra, row) in enumerate(zip(barras, df_rangos_dif.itertuples())):
    ax.text(barra.get_x() + barra.get_width()/2.,
            barra.get_height() + df_rangos_dif['Count'].max()*0.01,
            f'{row.Pct:.1f}%\n({int(row.Count)})',
            ha='center', va='bottom', fontsize=11, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.8, edgecolor='black'))

etiquetas = [f"{r['Range']}\n({r['Lower']:.1f}→{r['Upper']:.1f}mm)"
             for _, r in df_rangos_dif.iterrows()]
ax.set_xticks(range(len(etiquetas)))
ax.set_xticklabels(etiquetas, rotation=45, ha='right', fontsize=10, fontweight='bold')
ax.set_xlabel('Displacement Ranges (mm)', fontsize=13, fontweight='bold')
ax.set_ylabel('Record Count', fontsize=13, fontweight='bold')
ax.set_title(
    f'Displacement Distribution (Differences) — {sensor} {config["axis"]} | {config["zone"]}\n'
    f'Monitoring data — April 2025 to February 2026\n'
    f'Structural Displacements Recorded',
    fontsize=14, fontweight='bold', pad=20)
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(df_rangos_dif['Count'].max(), color='gray', linestyle='--', linewidth=1, alpha=0.5)
plt.subplots_adjust(top=0.95, bottom=0.15, left=0.08, right=0.98)
plt.tight_layout()
fname = f'{sensor}_02_displacement_ranges_barchart.png'
plt.savefig(os.path.join(sensor_dir, fname), dpi=300, bbox_inches='tight')
print(f'Exported: {fname}')
plt.show()

Exported: S12_02_displacement_ranges_barchart.png


In [13]:
# ============================================================
# PLOT 3: ABSOLUTE VALUES HISTOGRAM
# Shows distribution of raw laser distance readings.
# Percentile lines shown in absolute mm values.
# ============================================================
fig, ax = plt.subplots(figsize=(14, 7))

sns.histplot(datos, bins=50, edgecolor='black', kde=True,
             alpha=0.6, color=config['color'], label='Real values', ax=ax)

for valor, label, color, style in [
    (datos_min+p1,  'P1',    'red',    ':'),
    (datos_min+p5,  'P5',    'orange', '--'),
    (datos_min+Q1,  'Q1',    'green',  '-'),
    (datos_min+p50, 'Median','purple', '-'),
    (datos_min+Q3,  'Q3',    'green',  '-'),
    (datos_min+p95, 'P95',   'orange', '--'),
    (datos_min+p99, 'P99',   'red',    ':')
]:
    ax.axvline(valor, color=color, linestyle=style, linewidth=2.5, alpha=0.8, label=label)

margen = (datos.max() - datos.min()) * 0.05
ax.set_xlim(datos.min() - margen, datos.max() + margen)
ax.set_xticks([datos_min+p1, datos_min+p5, datos_min+Q1, datos_min+p50,
               datos_min+Q3, datos_min+p95, datos_min+p99])
ax.set_xticklabels([f'P1\n{datos_min+p1:.0f}mm', f'P5\n{datos_min+p5:.0f}mm',
                    f'Q1\n{datos_min+Q1:.0f}mm',  f'Median\n{datos_min+p50:.0f}mm',
                    f'Q3\n{datos_min+Q3:.0f}mm',  f'P95\n{datos_min+p95:.0f}mm',
                    f'P99\n{datos_min+p99:.0f}mm'],
                   fontsize=10, fontweight='bold', rotation=45, ha='right')
ax.set_xlabel('Absolute Values (mm)', fontsize=12, fontweight='bold')
ax.set_ylabel('Frequency', fontsize=12, fontweight='bold')
ax.set_title(
    f'Absolute Value Distribution — {sensor} {config["axis"]} | {config["zone"]}\n'
    f'Monitoring data — April 2025 to February 2026\n'
    f'Displacement Histogram (Real Values)',
    fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
ax.legend(loc='upper right', fontsize=9)
plt.tight_layout()
fname = f'{sensor}_03_absolute_values_histogram.png'
plt.savefig(os.path.join(sensor_dir, fname), dpi=300, bbox_inches='tight')
print(f'Exported: {fname}')
plt.show()

Exported: S12_03_absolute_values_histogram.png


In [14]:
# ============================================================
# PLOT 4: TEMPORAL EVOLUTION WITH PERCENTILE BANDS
# Shows time series with IQR reference lines and info box.
# The IQR band (Q1–Q3) represents the structural baseline.
# P95 represents the Yellow Alert threshold.
# ============================================================
rango   = movimiento_total
colchon = rango * 0.1
y_min   = datos_min - colchon
y_max   = datos_max + colchon

fig, ax = plt.subplots(figsize=(16, 7))

ax.plot(df_limpio['Timestamp'], df_limpio[sensor],
        linewidth=1.5, alpha=0.8, color=config['color'], label='Temporal evolution')

ax.set_ylim(y_min, y_max)

for valor, label, color in [
    (datos_min+p5,  f'P5: {datos_min+p5:.2f}',     'yellow'),
    (datos_min+Q1,  f'Q1: {datos_min+Q1:.2f}',     'green'),
    (datos_min+p50, f'Median: {datos_min+p50:.2f}', 'purple'),
    (datos_min+Q3,  f'Q3: {datos_min+Q3:.2f}',     'green'),
    (datos_min+p95, f'P95: {datos_min+p95:.2f}',   'yellow'),
]:
    ax.axhline(valor, color=color, linestyle='--', linewidth=3, alpha=0.7, label=label)

ax.set_title(
    f'Temporal Evolution — {sensor} {config["axis"]} | {config["zone"]}\n'
    f'Monitoring data — April 2025 to February 2026',
    fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel('Timestamp', fontsize=12, fontweight='bold')
ax.set_ylabel('Values (mm)', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11, loc='upper left', framealpha=0.95)

info_text = (
    f"INFORMATION {sensor} {config['axis']} | {config['zone']}\n"
    f"  Period: {df_limpio['Timestamp'].min()} to {df_limpio['Timestamp'].max()}\n"
    f"  Total records: {len(df_limpio)}\n"
    f"  Min: {datos_min:.2f} | Max: {datos_max:.2f}\n"
    f"  Mean: {datos.mean():.2f} | Std. Dev.: {datos.std():.2f}\n"
    f"  Range: {rango:.2f} mm"
)
ax.text(0.05, -0.25, info_text, transform=ax.transAxes, fontsize=10,
        verticalalignment='top', horizontalalignment='left',
        bbox=dict(boxstyle='round', facecolor='#E8E8E8', alpha=0.9),
        family='monospace', fontweight='bold')

plt.subplots_adjust(bottom=0.20)
plt.tight_layout()
fname = f'{sensor}_04_temporal_evolution.png'
plt.savefig(os.path.join(sensor_dir, fname), dpi=300, bbox_inches='tight')
print(f'Exported: {fname}')
plt.show()

Exported: S12_04_temporal_evolution.png


In [15]:
# ============================================================
# PLOT 5: TEMPORAL EVOLUTION vs ENVIRONMENTAL VARIABLES
# Normalized overlay using MinMaxScaler (range 0–10).
# Visual validation of Pearson correlations from notebook 04.
# ============================================================
df_norm = df_limpio.copy()

scaler_s = MinMaxScaler(feature_range=(0, 10))
df_norm[sensor] = scaler_s.fit_transform(df_limpio[[sensor]])

for var, color in [('Temp.', 'red'), ('Hum.', 'lightblue'), ('Pres.', 'lightgreen')]:
    if var in df_limpio.columns:
        scaler_v = MinMaxScaler(feature_range=(0, 10))
        df_norm[var] = scaler_v.fit_transform(df_limpio[[var]].fillna(df_limpio[var].mean()))

plt.figure(figsize=(40, 6))
plt.plot(df_norm['Timestamp'], df_norm[sensor],
         label=sensor, linewidth=2.5, color=config['color'], alpha=0.9)
plt.plot(df_norm['Timestamp'], df_norm['Temp.'],
         label='Temp. (normalized)', linewidth=1.5, alpha=0.4, color='red')
plt.plot(df_norm['Timestamp'], df_norm['Hum.'],
         label='Hum. (normalized)', linewidth=1.5, alpha=0.6, color='lightblue')
plt.plot(df_norm['Timestamp'], df_norm['Pres.'],
         label='Pres. (normalized)', linewidth=1.5, alpha=0.6, color='lightgreen')

plt.title(
    f'{sensor} {config["axis"]} | {config["zone"]} vs Environmental Variables (Normalized 0–10)\n'
    f'Monitoring data — April 2025 to February 2026 — Clean data',
    fontsize=14, fontweight='bold')
plt.xlabel('Time')
plt.ylabel('Normalized Value (0–10)')
plt.legend(loc='upper right', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
fname = f'{sensor}_05_temporal_vs_environment.png'
plt.savefig(os.path.join(sensor_dir, fname), dpi=300, bbox_inches='tight')
print(f'Exported: {fname}')
plt.show()

Exported: S12_05_temporal_vs_environment.png


In [16]:
# ============================================================
# PLOT 6: CORRELATION MATRIX — sensor vs environment
# ============================================================
correlation_data = df_limpio[[sensor, 'Temp.', 'Hum.', 'Pres.']].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(correlation_data, annot=True, fmt='.2f',
            cmap=['#001F3F', '#FFFFFF', '#C41E3A'],
            center=0, square=True, linewidths=2,
            cbar_kws={'label': 'Correlation'})
plt.title(
    f'Correlation Matrix — {sensor} {config["axis"]} | {config["zone"]}\n'
    f'vs Environmental Variables — Clean data\n'
    f'Monitoring data — April 2025 to February 2026',
    fontsize=12, fontweight='bold')
plt.tight_layout()
fname = f'{sensor}_06_correlation_matrix.png'
plt.savefig(os.path.join(sensor_dir, fname), dpi=300, bbox_inches='tight')
print(f'Exported: {fname}')
plt.show()

# Print correlation values
print(f"\nPearson r — {sensor} vs:")
for var in ['Temp.', 'Hum.', 'Pres.']:
    r = correlation_data.loc[sensor, var]
    print(f"  {var:<8} r = {r:+.3f}")

Exported: S12_06_correlation_matrix.png

Pearson r — S12 vs:
  Temp.    r = +0.649
  Hum.     r = -0.613
  Pres.    r = -0.361


In [17]:
# ============================================================
# EXPORT: PERCENTILE RANGE TABLE (two sheets — absolute + relative)
# ============================================================

# Build relative displacement table
df_rangos_rel = df_rangos_dif[['Range','Lower','Upper','Count','Pct','Cum']].copy()
df_rangos_rel.columns = ['Range','Lower (mm)','Upper (mm)','Count','% of total','Cumulative %']

# Build absolute values table
rangos_abs = []
for _, row in df_rangos_dif.iterrows():
    rangos_abs.append({
        'Range'       : row['Range'],
        'Lower (mm)'  : round(datos_min + row['Lower'], 2),
        'Upper (mm)'  : round(datos_min + row['Upper'], 2),
        'Count'       : row['Count'],
        '% of total'  : round(row['Pct'], 2),
        'Cumulative %': round(row['Cum'], 2)
    })
df_rangos_abs = pd.DataFrame(rangos_abs)

fname = f'{sensor}_07_analysis_table.xlsx'
out_path = os.path.join(sensor_dir, fname)
with pd.ExcelWriter(out_path, engine='openpyxl') as writer:
    df_rangos_rel.to_excel(writer, sheet_name='Relative Displacement', index=False)
    df_rangos_abs.to_excel(writer, sheet_name='Absolute Values',       index=False)

print(f'Exported: {fname}')
print(f'\nRelative displacement table:')
print(df_rangos_rel.to_string(index=False))

Exported: S12_07_analysis_table.xlsx

Relative displacement table:
      Range  Lower (mm)  Upper (mm)  Count  % of total  Cumulative %
   Min → P1      0.0000      0.2500    125    3.854456      3.854456
  P1 → P2.5      0.2500      0.2500      0    0.000000      3.854456
  P2.5 → P5      0.2500      0.3900     39    1.202590      5.057046
   P5 → P10      0.3900      0.7500    172    5.303731     10.360777
   P10 → Q1      0.7500      1.2500   1018   31.390688     41.751465
   Q1 → P50      1.2500      1.6500    279    8.603145     50.354610
   P50 → Q3      1.6500      2.8200    802   24.730188     75.084798
   Q3 → P90      2.8200      4.2500    554   17.082948     92.167746
  P90 → P95      4.2500      4.7500     93    2.867715     95.035461
P95 → P97.5      4.7500      5.2500     84    2.590194     97.625655
P97.5 → P99      5.2500      6.0758     44    1.356768     98.982424
  P99 → Max      6.0758      6.9500     33    1.017576    100.000000


In [18]:
# ============================================================
# BATCH MODE — ALL 12 SENSORS
# Generates the complete visual suite for all 12 sensors.
# Outputs saved to ../outputs/figures/{sensor}/
# 7 files per sensor × 12 sensors = 84 files total
# ============================================================
import matplotlib
matplotlib.use('Agg')   # non-interactive backend — no popup windows

batch_summary = []

for sensor in SENSORS:
    cfg   = SENSOR_CONFIG[sensor]
    s_dir = os.path.join(FIGURES_PATH, sensor)
    os.makedirs(s_dir, exist_ok=True)

    print(f"\n{'='*60}")
    print(f"Processing: {sensor} {cfg['axis']} | {cfg['zone']}")
    print(f"{'='*60}")

    try:
        df = pd.read_csv(os.path.join(PROCESSED_DATA_PATH, f'clean_{sensor}.csv'))
        df['Timestamp'] = pd.to_datetime(df['Timestamp'])
    except FileNotFoundError:
        print(f'  clean_{sensor}.csv not found — skipping')
        continue

    d         = df[sensor].dropna().copy()
    d_min     = d.min()
    d_max     = d.max()
    d_mov     = d - d_min
    mov_total = d_max - d_min

    p1_b    = float(d_mov.quantile(0.01))
    p2_5_b  = float(d_mov.quantile(0.025))
    p5_b    = float(d_mov.quantile(0.05))
    p10_b   = float(d_mov.quantile(0.10))
    Q1_b    = float(d_mov.quantile(0.25))
    p50_b   = float(d_mov.quantile(0.50))
    Q3_b    = float(d_mov.quantile(0.75))
    p90_b   = float(d_mov.quantile(0.90))
    p95_b   = float(d_mov.quantile(0.95))
    p97_5_b = float(d_mov.quantile(0.975))
    p99_b   = float(d_mov.quantile(0.99))

    # ── Plot 01: Relative displacement histogram ──────────────
    fig, ax = plt.subplots(figsize=(14, 7))
    ax.hist(d_mov, bins=50, edgecolor='black', alpha=0.6, color=cfg['color'])
    for v, lbl, col, sty in [
        (p1_b,  'P1',    'red',    ':'),
        (p5_b,  'P5',    'orange', '--'),
        (Q1_b,  'Q1',    'green',  '-'),
        (p50_b, 'Median','purple', '-'),
        (Q3_b,  'Q3',    'green',  '-'),
        (p95_b, 'P95',   'orange', '--'),
        (p99_b, 'P99',   'red',    ':')
    ]:
        ax.axvline(v, color=col, linestyle=sty, linewidth=2.5, alpha=0.8, label=lbl)
    margen = (d_mov.max() - d_mov.min()) * 0.05
    ax.set_xlim(d_mov.min() - margen, d_mov.max() + margen)
    ax.set_xticks([p1_b, p5_b, Q1_b, p50_b, Q3_b, p95_b, p99_b])
    ax.set_xticklabels([f'P1\n{p1_b:.1f}mm', f'P5\n{p5_b:.1f}mm', f'Q1\n{Q1_b:.1f}mm',
                        f'Median\n{p50_b:.1f}mm', f'Q3\n{Q3_b:.1f}mm',
                        f'P95\n{p95_b:.1f}mm', f'P99\n{p99_b:.1f}mm'],
                       fontsize=10, fontweight='bold')
    ax.set_xlabel('Relative Displacement (mm)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Frequency', fontsize=12, fontweight='bold')
    ax.set_title(f'Displacement Distribution — {sensor} {cfg["axis"]} | {cfg["zone"]}\n'
                 f'Monitoring data — April 2025 to February 2026',
                 fontsize=13, fontweight='bold')
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig(os.path.join(s_dir, f'{sensor}_01_displacement_distribution.png'),
                dpi=300, bbox_inches='tight')
    plt.close()
    print(f'  01 done')

    # ── Plot 02: Displacement ranges bar chart ────────────────
    rangos_data = []
    rango_1 = len(d_mov[d_mov <= p1_b])
    rangos_data.append({'Range': 'Min → P1', 'Lower': 0.0, 'Upper': p1_b,
                        'Count': rango_1, 'Pct': rango_1/len(d_mov)*100,
                        'Cum': rango_1/len(d_mov)*100})
    intermedios = [
        ('P1',p1_b,'P2.5',p2_5_b),('P2.5',p2_5_b,'P5',p5_b),('P5',p5_b,'P10',p10_b),
        ('P10',p10_b,'Q1',Q1_b),('Q1',Q1_b,'P50',p50_b),('P50',p50_b,'Q3',Q3_b),
        ('Q3',Q3_b,'P90',p90_b),('P90',p90_b,'P95',p95_b),
        ('P95',p95_b,'P97.5',p97_5_b),('P97.5',p97_5_b,'P99',p99_b)
    ]
    cum = rango_1
    for n_inf, v_inf, n_sup, v_sup in intermedios:
        n = len(d_mov[(d_mov > v_inf) & (d_mov <= v_sup)])
        cum += n
        rangos_data.append({'Range': f'{n_inf} → {n_sup}', 'Lower': v_inf, 'Upper': v_sup,
                            'Count': n, 'Pct': n/len(d_mov)*100, 'Cum': cum/len(d_mov)*100})
    n_final = len(d_mov[d_mov > p99_b])
    rangos_data.append({'Range': 'P99 → Max', 'Lower': p99_b, 'Upper': mov_total,
                        'Count': n_final, 'Pct': n_final/len(d_mov)*100, 'Cum': 100.0})
    df_rangos_dif = pd.DataFrame(rangos_data)

    colores = []
    for i, r in enumerate(df_rangos_dif['Range']):
        if i == 0 or i == len(df_rangos_dif) - 1:
            colores.append('#ff6b6b')
        elif 'P1' in r or 'P99' in r or 'P97.5' in r or 'P2.5' in r:
            colores.append('#ffa500')
        elif 'Q1' in r or 'P50' in r or 'Q3' in r:
            colores.append('#2ecc71')
        else:
            colores.append(cfg['color'])

    fig, ax = plt.subplots(figsize=(20, 10))
    barras = ax.bar(range(len(df_rangos_dif)), df_rangos_dif['Count'],
                    color=colores, edgecolor='black', linewidth=1.5, alpha=0.85)
    for barra, row in zip(barras, df_rangos_dif.itertuples()):
        ax.text(barra.get_x() + barra.get_width()/2.,
                barra.get_height() + df_rangos_dif['Count'].max()*0.01,
                f'{row.Pct:.1f}%\n({int(row.Count)})',
                ha='center', va='bottom', fontsize=11, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.8, edgecolor='black'))
    etiquetas = [f"{r['Range']}\n({r['Lower']:.1f}→{r['Upper']:.1f}mm)"
                 for _, r in df_rangos_dif.iterrows()]
    ax.set_xticks(range(len(etiquetas)))
    ax.set_xticklabels(etiquetas, rotation=45, ha='right', fontsize=10, fontweight='bold')
    ax.set_xlabel('Displacement Ranges (mm)', fontsize=13, fontweight='bold')
    ax.set_ylabel('Record Count', fontsize=13, fontweight='bold')
    ax.set_title(f'Displacement Ranges — {sensor} {cfg["axis"]} | {cfg["zone"]}\n'
                 f'Monitoring data — April 2025 to February 2026',
                 fontsize=14, fontweight='bold', pad=20)
    ax.grid(True, alpha=0.3, axis='y')
    plt.subplots_adjust(top=0.95, bottom=0.15, left=0.08, right=0.98)
    plt.tight_layout()
    plt.savefig(os.path.join(s_dir, f'{sensor}_02_displacement_ranges_barchart.png'),
                dpi=300, bbox_inches='tight')
    plt.close()
    print(f'  02 done')

    # ── Plot 03: Absolute values histogram ────────────────────
    fig, ax = plt.subplots(figsize=(14, 7))
    sns.histplot(d, bins=50, edgecolor='black', kde=True,
                 alpha=0.6, color=cfg['color'], label='Real values', ax=ax)
    for v, lbl, col, sty in [
        (d_min+p1_b,  'P1',    'red',    ':'),
        (d_min+p5_b,  'P5',    'orange', '--'),
        (d_min+Q1_b,  'Q1',    'green',  '-'),
        (d_min+p50_b, 'Median','purple', '-'),
        (d_min+Q3_b,  'Q3',    'green',  '-'),
        (d_min+p95_b, 'P95',   'orange', '--'),
        (d_min+p99_b, 'P99',   'red',    ':')
    ]:
        ax.axvline(v, color=col, linestyle=sty, linewidth=2.5, alpha=0.8, label=lbl)
    margen = (d.max() - d.min()) * 0.05
    ax.set_xlim(d.min() - margen, d.max() + margen)
    ax.set_xticks([d_min+p1_b, d_min+p5_b, d_min+Q1_b, d_min+p50_b,
                   d_min+Q3_b, d_min+p95_b, d_min+p99_b])
    ax.set_xticklabels([f'P1\n{d_min+p1_b:.0f}mm', f'P5\n{d_min+p5_b:.0f}mm',
                        f'Q1\n{d_min+Q1_b:.0f}mm',  f'Median\n{d_min+p50_b:.0f}mm',
                        f'Q3\n{d_min+Q3_b:.0f}mm',  f'P95\n{d_min+p95_b:.0f}mm',
                        f'P99\n{d_min+p99_b:.0f}mm'],
                       fontsize=10, fontweight='bold', rotation=45, ha='right')
    ax.set_xlabel('Absolute Values (mm)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Frequency', fontsize=12, fontweight='bold')
    ax.set_title(f'Absolute Value Distribution — {sensor} {cfg["axis"]} | {cfg["zone"]}\n'
                 f'Monitoring data — April 2025 to February 2026',
                 fontsize=13, fontweight='bold')
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig(os.path.join(s_dir, f'{sensor}_03_absolute_values_histogram.png'),
                dpi=300, bbox_inches='tight')
    plt.close()
    print(f'  03 done')

    # ── Plot 04: Temporal evolution ───────────────────────────
    colchon = mov_total * 0.1
    fig, ax = plt.subplots(figsize=(16, 7))
    ax.plot(df['Timestamp'], df[sensor], linewidth=1.5, alpha=0.8,
            color=cfg['color'], label='Temporal evolution')
    ax.set_ylim(d_min - colchon, d_max + colchon)
    for v, lbl, col in [
        (d_min+p5_b,  f'P5:{d_min+p5_b:.1f}',    'yellow'),
        (d_min+Q1_b,  f'Q1:{d_min+Q1_b:.1f}',    'green'),
        (d_min+p50_b, f'Median:{d_min+p50_b:.1f}','purple'),
        (d_min+Q3_b,  f'Q3:{d_min+Q3_b:.1f}',    'green'),
        (d_min+p95_b, f'P95:{d_min+p95_b:.1f}',  'yellow')
    ]:
        ax.axhline(v, color=col, linestyle='--', linewidth=2.5, alpha=0.7, label=lbl)
    info = (f"INFO {sensor} | Period: {df['Timestamp'].min()} to {df['Timestamp'].max()}\n"
            f"Total records: {len(df)} | Min: {d_min:.2f} | Max: {d_max:.2f}\n"
            f"Mean: {d.mean():.2f} | Std.Dev: {d.std():.2f} | Range: {mov_total:.2f}mm")
    ax.text(0.05, -0.22, info, transform=ax.transAxes, fontsize=9,
            va='top', ha='left', family='monospace', fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='#E8E8E8', alpha=0.9))
    ax.set_title(f'Temporal Evolution — {sensor} {cfg["axis"]} | {cfg["zone"]}\n'
                 f'Monitoring data — April 2025 to February 2026',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('Timestamp', fontsize=12)
    ax.set_ylabel('Values (mm)', fontsize=12)
    ax.legend(fontsize=10, loc='upper left')
    ax.grid(True, alpha=0.3)
    plt.subplots_adjust(bottom=0.18)
    plt.tight_layout()
    plt.savefig(os.path.join(s_dir, f'{sensor}_04_temporal_evolution.png'),
                dpi=300, bbox_inches='tight')
    plt.close()
    print(f'  04 done')

    # ── Plot 05: Temporal vs environmental overlay ────────────
    df_norm = df.copy()
    scaler_s = MinMaxScaler(feature_range=(0, 10))
    df_norm[sensor] = scaler_s.fit_transform(df[[sensor]])
    for var in ['Temp.', 'Hum.', 'Pres.']:
        scaler_v = MinMaxScaler(feature_range=(0, 10))
        df_norm[var] = scaler_v.fit_transform(df[[var]].fillna(df[var].mean()))
    fig, ax = plt.subplots(figsize=(40, 6))
    ax.plot(df_norm['Timestamp'], df_norm[sensor],
            label=sensor, linewidth=2.5, color=cfg['color'], alpha=0.9)
    ax.plot(df_norm['Timestamp'], df_norm['Temp.'],
            label='Temp. (normalized)', linewidth=1.5, alpha=0.4, color='red')
    ax.plot(df_norm['Timestamp'], df_norm['Hum.'],
            label='Hum. (normalized)', linewidth=1.5, alpha=0.6, color='lightblue')
    ax.plot(df_norm['Timestamp'], df_norm['Pres.'],
            label='Pres. (normalized)', linewidth=1.5, alpha=0.6, color='lightgreen')
    ax.set_title(f'{sensor} {cfg["axis"]} | {cfg["zone"]} vs Environmental Variables (Normalized 0–10)\n'
                 f'Monitoring data — April 2025 to February 2026 — Clean data',
                 fontsize=14, fontweight='bold')
    ax.set_xlabel('Time')
    ax.set_ylabel('Normalized Value (0–10)')
    ax.legend(loc='upper right', fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(s_dir, f'{sensor}_05_temporal_vs_environment.png'),
                dpi=300, bbox_inches='tight')
    plt.close()
    print(f'  05 done')

    # ── Plot 06: Correlation matrix ───────────────────────────
    corr_data = df[[sensor, 'Temp.', 'Hum.', 'Pres.']].corr()
    plt.figure(figsize=(8, 6))
    sns.heatmap(corr_data, annot=True, fmt='.2f',
                cmap=['#001F3F', '#FFFFFF', '#C41E3A'],
                center=0, square=True, linewidths=2,
                cbar_kws={'label': 'Correlation'})
    plt.title(f'Correlation Matrix — {sensor} {cfg["axis"]} | {cfg["zone"]}\n'
              f'vs Environmental Variables — Clean data\n'
              f'Monitoring data — April 2025 to February 2026',
              fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(s_dir, f'{sensor}_06_correlation_matrix.png'),
                dpi=300, bbox_inches='tight')
    plt.close()
    print(f'  06 done')

    # ── Export 07: Percentile range table xlsx ────────────────
    df_rangos_rel = df_rangos_dif[['Range','Lower','Upper','Count','Pct','Cum']].copy()
    df_rangos_rel.columns = ['Range','Lower (mm)','Upper (mm)','Count','% of total','Cumulative %']
    rangos_abs = []
    for _, row in df_rangos_dif.iterrows():
        rangos_abs.append({
            'Range'       : row['Range'],
            'Lower (mm)'  : round(d_min + row['Lower'], 2),
            'Upper (mm)'  : round(d_min + row['Upper'], 2),
            'Count'       : row['Count'],
            '% of total'  : round(row['Pct'], 2),
            'Cumulative %': round(row['Cum'], 2)
        })
    df_rangos_abs = pd.DataFrame(rangos_abs)
    fname_xlsx = f'{sensor}_07_analysis_table.xlsx'
    with pd.ExcelWriter(os.path.join(s_dir, fname_xlsx), engine='openpyxl') as writer:
        df_rangos_rel.to_excel(writer, sheet_name='Relative Displacement', index=False)
        df_rangos_abs.to_excel(writer, sheet_name='Absolute Values',       index=False)
    print(f'  07 done')

    batch_summary.append({
        'Sensor'              : sensor,
        'Zone'                : cfg['zone'],
        'Axis'                : cfg['axis'],
        'Records'             : len(d),
        'Max displacement (mm)': round(mov_total, 2),
        'Mean (mm)'           : round(d_mov.mean(), 2),
        'Std Dev (mm)'        : round(d_mov.std(), 2),
        'CV (%)'              : round(d_mov.std()/d_mov.mean()*100, 2),
        'IQR range (mm)'      : f'{Q1_b:.2f}–{Q3_b:.2f}',
        'Output folder'       : s_dir
    })
    print(f'  ✓ {sensor} complete — 7 files saved')

# Restore interactive backend
matplotlib.use('TkAgg')

df_batch = pd.DataFrame(batch_summary)
print(f"\n{'='*100}")
print(f"BATCH COMPLETE — {len(batch_summary)} sensors processed")
print(f"{'='*100}\n")
print(df_batch.to_string(index=False))

df_batch.to_excel(os.path.join(TABLES_PATH, '05_batch_summary.xlsx'), index=False)
print(f'\nExported: 05_batch_summary.xlsx')


Processing: S01 (Diag.) | Zone A
  01 done
  02 done
  03 done
  04 done
  05 done
  06 done
  07 done
  ✓ S01 complete — 7 files saved

Processing: S02 (Hor.) | Zone A
  01 done
  02 done
  03 done
  04 done
  05 done
  06 done
  07 done
  ✓ S02 complete — 7 files saved

Processing: S03 (Diag.) | Zone A
  01 done
  02 done
  03 done
  04 done
  05 done
  06 done
  07 done
  ✓ S03 complete — 7 files saved

Processing: S04 (Hor.) | Zone A
  01 done
  02 done
  03 done
  04 done
  05 done
  06 done
  07 done
  ✓ S04 complete — 7 files saved

Processing: S05 (Diag.) | Zone B
  01 done
  02 done
  03 done
  04 done
  05 done
  06 done
  07 done
  ✓ S05 complete — 7 files saved

Processing: S06 (Hor.) | Zone B
  01 done
  02 done
  03 done
  04 done
  05 done
  06 done
  07 done
  ✓ S06 complete — 7 files saved

Processing: S07 (Diag.) | Zone B
  01 done
  02 done
  03 done
  04 done
  05 done
  06 done
  07 done
  ✓ S07 complete — 7 files saved

Processing: S08 (Hor.) | Zone B
  01 done
 

---
## Summary

Per sensor outputs (saved to `../outputs/figures/{sensor}/`):

| File | Content |
|------|---------|
| `{sensor}_01_displacement_distribution.png` | Relative displacement histogram with percentile bands |
| `{sensor}_02_displacement_ranges_barchart.png` | Bar chart by percentile range with counts and percentages |
| `{sensor}_03_absolute_values_histogram.png` | Raw absolute value histogram with KDE |
| `{sensor}_04_temporal_evolution.png` | Time series with IQR reference lines and info box |
| `{sensor}_05_temporal_vs_environment.png` | Normalized overlay vs Temperature, Humidity, Pressure |
| `{sensor}_06_correlation_matrix.png` | Pearson heatmap vs environmental variables |
| `{sensor}_07_analysis_table.xlsx` | Percentile range table (absolute + relative sheets) |

**Next:** `INFRASTRUCTURE_ANALYSIS_REPORT.md` — anonymized technical report with selected visualizations
and full engineering interpretation.